# 2. Silver — Nettoyage & Validation

Nettoyage et validation de la table bronze `workspace.projet_alpha.insurance_claim_fraud_bronze`, puis écriture dans `workspace.projet_alpha.insurance_claim_fraud_silver`.

Étapes, dans l'ordre :
1. **DATE** : cast `to_date()` + élimination des dates aberrantes
2. **INTEGER** : valeurs impossibles -> NULL ou imputation à 0
3. **DOUBLE** : négatifs -> NULL ou 0, arrondi à 2 décimales
4. **STRING** : TRIM + UPPER (INITCAP pour `customer_name`), NULL -> 'UNKNOWN'
5. **Déduplication** sur `claim_id` (ROW_NUMBER) + colonnes d'audit
6. **Écriture idempotente** : mode `overwrite`

In [0]:
# Paramètres du pipeline Silver
bronze_table = "workspace.projet_alpha.insurance_claim_fraud_bronze"
silver_table = "workspace.projet_alpha.insurance_claim_fraud_silver"
table_name = bronze_table   # source des requêtes SQL ci-dessous

In [0]:
# 1) DATE : cast en type date + élimination des dates aberrantes hors des ranges métier
#    to_date() renvoie NULL si mal formaté -> la ligne est éliminée par le BETWEEN
#    claim_date / incident_date : >= 2021-01-01 | policy_start_date : >= 2016-02-11
table_silver = spark.sql(f"""
    SELECT * EXCEPT (claim_date, incident_date, policy_start_date),
           to_date(claim_date)        AS claim_date,
           to_date(incident_date)     AS incident_date,
           to_date(policy_start_date) AS policy_start_date
    FROM {table_name}
    WHERE to_date(claim_date)        BETWEEN DATE '2021-01-01' AND current_date()
      AND to_date(incident_date)     BETWEEN DATE '2021-01-01' AND current_date()
      AND to_date(policy_start_date) BETWEEN DATE '2016-02-11' AND current_date()
""")

nb_rows_raw = spark.table(bronze_table).count()
nb_rows_silver = table_silver.count()
print(f"Bronze : {nb_rows_raw} lignes | après nettoyage des dates : {nb_rows_silver} | éliminées : {nb_rows_raw - nb_rows_silver}")

In [0]:
# 2) INTEGER : valeurs impossibles -> NULL ; colonnes techniques/comptables -> imputées à 0 ;
#    colonnes métier essentielles (age, vehicle_year, fraud_risk_score) -> NULL conservé
table_silver.createOrReplaceTempView("silver_tmp")

table_silver = spark.sql("""
    SELECT * EXCEPT (claim_duration_days, age, credit_score, years_with_company, previous_claims,
                     policy_duration_years, witnesses, injuries, vehicle_year, vehicle_age,
                     mileage, fraud_risk_score),
           COALESCE(CASE WHEN claim_duration_days   >= 0 THEN claim_duration_days   END, 0) AS claim_duration_days,
           CASE         WHEN age              BETWEEN 18 AND 100  THEN age              END AS age,
           CASE         WHEN credit_score     BETWEEN 300 AND 850 THEN credit_score     END AS credit_score,
           COALESCE(CASE WHEN years_with_company  >= 0 THEN years_with_company  END, 0) AS years_with_company,
           COALESCE(CASE WHEN previous_claims     >= 0 THEN previous_claims     END, 0) AS previous_claims,
           COALESCE(CASE WHEN policy_duration_years >= 0 THEN policy_duration_years END, 0) AS policy_duration_years,
           COALESCE(CASE WHEN witnesses         >= 0 THEN witnesses         END, 0) AS witnesses,
           COALESCE(CASE WHEN injuries          >= 0 THEN injuries          END, 0) AS injuries,
           CASE         WHEN vehicle_year   BETWEEN 1980 AND year(current_date()) THEN vehicle_year END AS vehicle_year,
           COALESCE(CASE WHEN vehicle_age       >= 0 THEN vehicle_age       END, 0) AS vehicle_age,
           COALESCE(CASE WHEN mileage           >= 0 THEN mileage           END, 0) AS mileage,
           CASE         WHEN fraud_risk_score BETWEEN 0 AND 100 THEN fraud_risk_score END AS fraud_risk_score
    FROM silver_tmp
""")
print(f"Lignes après traitement INTEGER : {table_silver.count()}")

In [0]:
# 3) DOUBLE : négatifs -> NULL ou 0, arrondi à 2 décimales (colonnes déjà typées DOUBLE dans bronze)
table_silver.createOrReplaceTempView("silver_tmp")

table_silver = spark.sql("""
    SELECT * EXCEPT (claim_amount, approved_amount, deductible, annual_income,
                     premium_amount, coverage_amount, vehicle_value, repair_cost,
                     medical_cost, towing_cost, rental_vehicle_cost, payout_amount),
           ROUND(CASE         WHEN claim_amount        >= 0 THEN claim_amount        END, 2) AS claim_amount,
           ROUND(CASE         WHEN approved_amount     >= 0 THEN approved_amount     END, 2) AS approved_amount,
           ROUND(COALESCE(CASE WHEN deductible          >= 0 THEN deductible          END, 0), 2) AS deductible,
           ROUND(CASE         WHEN annual_income       >= 0 THEN annual_income       END, 2) AS annual_income,
           ROUND(COALESCE(CASE WHEN premium_amount      >= 0 THEN premium_amount      END, 0), 2) AS premium_amount,
           ROUND(COALESCE(CASE WHEN coverage_amount     >= 0 THEN coverage_amount     END, 0), 2) AS coverage_amount,
           ROUND(CASE         WHEN vehicle_value       >= 0 THEN vehicle_value       END, 2) AS vehicle_value,
           ROUND(COALESCE(CASE WHEN repair_cost          >= 0 THEN repair_cost          END, 0), 2) AS repair_cost,
           ROUND(COALESCE(CASE WHEN medical_cost         >= 0 THEN medical_cost         END, 0), 2) AS medical_cost,
           ROUND(COALESCE(CASE WHEN towing_cost          >= 0 THEN towing_cost          END, 0), 2) AS towing_cost,
           ROUND(COALESCE(CASE WHEN rental_vehicle_cost  >= 0 THEN rental_vehicle_cost  END, 0), 2) AS rental_vehicle_cost,
           ROUND(COALESCE(CASE WHEN payout_amount        >= 0 THEN payout_amount        END, 0), 2) AS payout_amount
    FROM silver_tmp
""")
print(f"Lignes après traitement DOUBLE : {table_silver.count()}")

In [0]:
# 4) STRING : TRIM + casse standardisée (UPPER pour codes/catégories/flags, INITCAP pour customer_name)
#    colonnes métier essentielles (IDs, customer_name) -> NULL conservé ; catégorielles -> 'UNKNOWN'
table_silver.createOrReplaceTempView("silver_tmp")

string_columns = [
    "claim_id", "policy_id", "customer_id", "customer_name", "claim_status",
    "gender", "marital_status", "occupation", "education_level", "policy_type",
    "deductible_plan", "payment_method", "policy_active", "incident_type",
    "accident_severity", "incident_location", "weather_condition", "police_report",
    "hospital_visit", "property_damage", "vehicle_damage", "vehicle_make",
    "vehicle_model", "ownership_type", "suspicious_documents",
    "claim_submitted_at_night", "multiple_claims_last_year", "delayed_reporting",
    "inconsistent_statements", "gps_location_match", "repair_shop_verified",
    "identity_verified", "claim_history_risk", "total_loss", "fraudulent"
]
string_columns_keep_null = ["claim_id", "policy_id", "customer_id", "customer_name"]

def clean_expr(col):
    inner = f"INITCAP(TRIM({col}))" if col == "customer_name" else f"UPPER(TRIM({col}))"
    return f"COALESCE({inner}, 'UNKNOWN')" if col not in string_columns_keep_null else inner

string_cleaning_select = ",\n           ".join(f"{clean_expr(c)} AS {c}" for c in string_columns)

table_silver = spark.sql(f"""
    SELECT * EXCEPT ({', '.join(string_columns)}),
           {string_cleaning_select}
    FROM silver_tmp
""")
print(f"Lignes après traitement STRING : {table_silver.count()}")

In [0]:
# 5) Déduplication sur claim_id (ligne la plus récente conservée) + colonnes d'audit
#    NB : les lignes avec claim_id NULL sont conservées (pas de clé fiable pour les comparer)
from datetime import datetime

nb_rows_before_dedup = table_silver.count()
table_silver.createOrReplaceTempView("silver_tmp")

table_silver = spark.sql("""
    SELECT * EXCEPT (rn)
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY claim_id
                   ORDER BY claim_date DESC, policy_id DESC
               ) AS rn
        FROM silver_tmp
    )
    WHERE rn = 1 OR claim_id IS NULL
""")

batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")
table_silver.createOrReplaceTempView("silver_tmp")
table_silver = spark.sql(f"""
    SELECT *,
           current_timestamp() AS silver_ingestion_timestamp,
           '{bronze_table}'     AS source_bronze_table,
           '{batch_id}'         AS silver_batch_id
    FROM silver_tmp
""")

nb_rows_after_dedup = table_silver.count()
print(f"Avant dédup : {nb_rows_before_dedup} | après : {nb_rows_after_dedup} | doublons éliminés : {nb_rows_before_dedup - nb_rows_after_dedup}")

# 6) Écriture idempotente : mode "overwrite" -> relancer le notebook ne duplique pas les lignes
table_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)

print(f"Table Silver sauvegardée : {nb_rows_after_dedup} lignes dans {silver_table}")

# Contrôle de cohérence final : lignes Bronze vs lignes Silver
nb_bronze = spark.table(bronze_table).count()
nb_silver = spark.table(silver_table).count()
print(f"Bronze : {nb_bronze} lignes | Silver : {nb_silver} lignes | écart : {nb_bronze - nb_silver} (dates aberrantes + doublons)")

### Récapitulatif des traitements Silver

| Type | Traitement | Fonctions SQL |
|---|---|---|
| DATE | Cast strict + filtrage des dates aberrantes | `to_date()`, `BETWEEN DATE ... AND current_date()` |
| INTEGER | Valeurs impossibles -> NULL ; colonnes techniques -> 0 | `CASE WHEN ... THEN ... END`, `COALESCE(col, 0)` |
| DOUBLE | Négatifs -> NULL ou 0 ; arrondi à 2 décimales | `ROUND(col, 2)`, `COALESCE` |
| STRING | Trim, casse standardisée, imputation des catégorielles | `TRIM()`, `UPPER()`, `INITCAP()`, `COALESCE(col, 'UNKNOWN')` |
| Global | Réécriture ciblée des colonnes transformées | `SELECT * EXCEPT (...)` |
| Dédup | Une ligne par claim_id (la plus récente) | `ROW_NUMBER() OVER (PARTITION BY claim_id ORDER BY claim_date DESC)` |

Ordre d'exécution : DATE -> INTEGER -> DOUBLE -> STRING -> déduplication -> écriture en mode overwrite.